# pytest

실제 결함을 실패하는 테스트로 확인하고, 수정 뒤 통과하는 테스트로 어떻게 남길까요?

## 실행 환경 준비

Python 3.12와 pytest 9.1.1을 사용합니다. 테스트 파일은 임시 폴더에 만들며, pytest는 현재 커널의 Python으로 실행합니다.

In [1]:
%pip install -q pytest==9.1.1

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path
from subprocess import CompletedProcess, run
from sys import executable
from tempfile import TemporaryDirectory

workspace = TemporaryDirectory(prefix="ch02_08_")
test_dir = Path(workspace.name)
pytest_env = {**os.environ, "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1"}
print("pytest 실습 폴더 준비 완료")

pytest 실습 폴더 준비 완료


## 기본 실행

가장 작은 테스트는 준비한 값을 실행하고 `assert`로 기대값과 비교합니다. 종료 코드 0은 모든 테스트가 통과했다는 뜻입니다.

In [3]:
def run_pytest(folder: Path, target: str) -> CompletedProcess[str]:
    return run(
        [executable, "-m", "pytest", "-q", "--tb=short", target],
        cwd=folder, env=pytest_env, capture_output=True, text=True
    )

basic_test = '''def test_total() -> None:
    assert 1200 + 800 == 2000
'''
(test_dir / "test_basic.py").write_text(basic_test, encoding="utf-8")
basic_result = run_pytest(test_dir, "test_basic.py")
print(basic_result.stdout.strip())
assert basic_result.returncode == 0

.                                                                        [100%]
1 passed in 0.00s


## 중간 결과 관찰

매개변수화는 같은 검사 규칙에 여러 입력과 기대값을 연결합니다. 세 사례가 각각 하나의 테스트로 실행됩니다.

In [4]:
param_test = '''import pytest

@pytest.mark.parametrize(("price", "count", "expected"), [
    (1000, 1, 1000), (1200, 2, 2400), (500, 0, 0),
])
def test_line_total(price: int, count: int, expected: int) -> None:
    assert price * count == expected
'''
(test_dir / "test_param.py").write_text(param_test, encoding="utf-8")
param_result = run_pytest(test_dir, "test_param.py")
print(param_result.stdout.strip())
assert param_result.returncode == 0

...                                                                      [100%]
3 passed in 0.00s


## 값 하나를 바꿔 비교

매개변수 목록에 할인된 가격 사례 하나를 추가하면 전체 테스트 수가 늘어납니다.

In [5]:
expanded = param_test.replace("(500, 0, 0),", "(500, 0, 0), (900, 3, 2700),")
(test_dir / "test_param.py").write_text(expanded, encoding="utf-8")
expanded_result = run_pytest(test_dir, "test_param.py")
print(expanded_result.stdout.strip())
assert expanded_result.returncode == 0

....                                                                     [100%]
4 passed in 0.00s


## 실패 재현

할인 함수가 10% 할인 뒤 100원을 잘못 더하는 결함을 넣습니다. 같은 입력 10,000원에 기대값 9,000원을 검사하여 실제 실패를 확인합니다.

In [6]:
buggy_test = '''def discounted(price: int) -> int:
    return int(price * 0.9) + 100

def test_discounted() -> None:
    assert discounted(10000) == 9000
'''
(test_dir / "test_discount.py").write_text(buggy_test, encoding="utf-8")
failed = run_pytest(test_dir, "test_discount.py")
print(failed.stdout.strip())
print("종료 코드:", failed.returncode)
assert failed.returncode == 1

F                                                                        [100%]
=================================== FAILURES ===================================
_______________________________ test_discounted ________________________________
test_discount.py:5: in test_discounted
    assert discounted(10000) == 9000
E   assert 9100 == 9000
E    +  where 9100 = discounted(10000)
=========================== short test summary info ============================
FAILED test_discount.py::test_discounted - assert 9100 == 9000
1 failed in 0.01s
종료 코드: 1


## 수정

같은 함수와 입력에서 잘못 더한 100을 제거합니다. 테스트 코드의 기대값은 바꾸지 않고 다시 실행합니다.

In [7]:
fixed_test = buggy_test.replace("int(price * 0.9) + 100", "int(price * 0.9)")
(test_dir / "test_discount.py").write_text(fixed_test, encoding="utf-8")
passed = run_pytest(test_dir, "test_discount.py")
print(passed.stdout.strip())
print("종료 코드:", passed.returncode)
assert passed.returncode == 0

.                                                                        [100%]
1 passed in 0.00s
종료 코드: 0
